# Pegasus Workflow Visualizer — Demo

This notebook demonstrates the workflow visualizer widget rendering a Pegasus DAG
with real-time state updates from `workflow-events.jsonl`.

## Source Modes

The visualizer supports three data source modes, shown as a badge in the toolbar:

| Mode       | Badge    | Description                                              |
|------------|----------|----------------------------------------------------------|
| **Static** | `STATIC` | DAG structure only — no live event feed                  |
| **Live**   | `LIVE`   | Polling a local `workflow-events.jsonl` file             |
| **SSH**    | `SSH`    | Fetching events from a remote host over SSH              |

In [1]:
from workflow_visualizer import WorkflowVisualizerWidget

## 1. Static DAG from workflow.yml

Render the earthquake workflow DAG without live events.
The toolbar shows the **STATIC** badge (gray) — no event source is connected.

In [2]:
# Static DAG — just the graph structure
w = WorkflowVisualizerWidget(
    #workflow_path="path/to/workflow.yml"
    workflow_path="/Users/stealey/GitHub/pegasusai/example-workflows/earthquake-workflow/workflow.yml"
)
w

## 2. Live DAG with local event feed

Feed a local JSONL file to see nodes animate through states.
The toolbar shows the **LIVE** badge (green, pulsing) — polling the local file.

In [ ]:
# DAG + live event updates
w2 = WorkflowVisualizerWidget(
    workflow_path="path/to/workflow.yml",
    jsonl_path="path/to/workflow-events.jsonl",
    poll_interval=2.0,
)
w2

## 3. Remote SSH workflow (FABRIC testbed example)

Connect to a remote Pegasus submit node over SSH to monitor a running workflow.
The toolbar shows the **SSH** badge (purple, pulsing) with the remote host on hover.

### FABRIC SSH prerequisites

FABRIC nodes are accessed through a bastion host. You need:

1. **Slice key** — the SSH private key generated for your FABRIC slice
   (typically at `~/work/fabric_config/slice_key` on FABRIC JupyterHub,
   or wherever you saved it locally)
2. **SSH config** — a config file that routes through the FABRIC bastion:

```
# ~/.ssh/fabric-ssh-config

UserKnownHostsFile /dev/null
StrictHostKeyChecking no
ServerAliveInterval 120

Host bastion.fabric-testbed.net
    User <your-fabric-username>
    IdentityFile ~/.ssh/fabric-bastion-key
    ForwardAgent yes

Host pegasus-submit
    User ubuntu
    HostName <submit-node-ip>
    ProxyJump bastion.fabric-testbed.net
    IdentityFile ~/.ssh/my-sliver-key
```

With this config, `ssh pegasus-submit` tunnels through the bastion automatically.

In [ ]:
# SSH remote workflow — FABRIC testbed
#
# Before running: ensure workflow-monitor is serving on the submit node:
#   ssh pegasus-submit "uv run workflow-monitor --serve /home/ubuntu/my-workflow"
#
# Option A: Using SSH config with a host alias (recommended for FABRIC)
w3 = WorkflowVisualizerWidget(
    workflow_path="/home/ubuntu/my-workflow/workflow.yml",
    remote_spec="pegasus-submit:/home/ubuntu/my-workflow/ubuntu/pegasus/wf/run0001/workflow-events.jsonl",
    ssh_config="~/.ssh/fabric-ssh-config",
    ssh_identity="~/.ssh/my-sliver-key",
    poll_interval=5.0,
)
w3

In [ ]:
# Option B: Using an IPv6 address directly (no SSH config alias)
# Useful when connecting from FABRIC JupyterHub to a node on the same slice
#
# w3_ipv6 = WorkflowVisualizerWidget(
#     remote_spec="ubuntu@[2001:db8::1]:/home/ubuntu/my-workflow/ubuntu/pegasus/wf/run0001/workflow-events.jsonl",
#     ssh_config="~/.ssh/fabric-ssh-config",
#     ssh_identity="~/.ssh/my-sliver-key",
#     poll_interval=5.0,
# )
# w3_ipv6

In [ ]:
# Option B: Using an IPv6 address directly (no SSH config alias)
# Useful when connecting from FABRIC JupyterHub to a node on the same slice
#
# w3_ipv6 = WorkflowVisualizerWidget(
#     remote_spec="ubuntu@[2001:db8::1]:/home/ubuntu/earthquake-workflow/ubuntu/pegasus/earthquake/run0001/workflow-events.jsonl",
#     poll_interval=5.0,
# )
# w3_ipv6

## 4. No workflow file — placeholder

When no workflow path is provided, a file chooser is shown in the viewport.
You can enter a path to `workflow.yml` interactively.

In [ ]:
# Placeholder — enter a workflow.yml path in the UI
w4 = WorkflowVisualizerWidget()
w4